# AOT-компиляция UNet SD1.5 под Google Tensor G5 (Colab)

Локально компиляция падает `INTERNAL`, т.к. потребительский CPU (Arrow Lake) **без AVX-512**.
Colab-рантайм — Xeon **с AVX-512**, поэтому компилятор Tensor тут работает.

## Порядок действий
1. Положи на Drive в `MyDrive/tensor_tpu/` свой SDK-тарбол **`litert_plugin_compiler.tar.gz`**.
2. **Сначала выполни Приложение внизу** (в чистом рантайме) — оно сгенерирует `unet.tflite` (fp32)
   прямо в Colab и положит на Drive. Так не нужно заливать гигабайты вручную.
3. Перезапусти рантайм (Runtime → Restart) и иди с ячейки 1.

Компилятор берёт **fp32**-модель и сам делает fp16 на TPU через флаг `truncation='half'`.

## Runtime
Хватает обычного **CPU high-RAM** (компиляция на x86-CPU, не GPU). Runtime → Change runtime type → CPU, High-RAM.

## 1. Монтируем Drive и задаём пути

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/tensor_tpu'   # <-- твоя папка на Drive
SDK_TARBALL = os.path.join(WORK, 'litert_plugin_compiler.tar.gz')
MODEL_TFLITE = os.path.join(WORK, 'unet.tflite')   # fp32, из Приложения
OUT_DIR = os.path.join(WORK, 'compiled')
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(SDK_TARBALL), f'нет SDK-тарбола: {SDK_TARBALL}'
print('SDK tarball:', os.path.getsize(SDK_TARBALL) // 1_000_000, 'MB')
print('model exists:', os.path.exists(MODEL_TFLITE))
print('CPU avx512:', 'avx512' in open('/proc/cpuinfo').read())  # должно быть True на Colab

## 2. Ставим AOT-окружение (только то, что нужно компилятору)
Точный dev20260518 из гугловского ноутбука удалён с pypi → берём ближайший `dev20260520`
(та же ABI). Предупреждения о version mismatch — ожидаемы, игнорируй.

In [ ]:
# На Colab убираем предустановленный tensorflow, чтобы не конфликтовал
!pip uninstall -y tensorflow tensorflow-cpu 2>/dev/null

!pip install -q ai-edge-litert-nightly==2.2.0.dev20260520

# SDK-обёртка берёт бинарь компилятора из нашего тарбола на Drive
import os
os.environ['GOOGLE_TENSOR_SDK_BETA'] = SDK_TARBALL
!GOOGLE_TENSOR_SDK_BETA="$SDK_TARBALL" pip install -q ai-edge-litert-sdk-google-tensor==2.1.5
print('готово — если pip просит перезапуск, Runtime → Restart, затем выполняй с ячейки 3')

## 3. Проверяем, что SDK и backend на месте

In [ ]:
import os
import ai_edge_litert, ai_edge_litert_sdk_google_tensor as sdk
print('litert:', ai_edge_litert.__version__)
print('sdk libs:', os.listdir(sdk.path_to_sdk_libs()))
from ai_edge_litert.aot.vendors import import_vendor
import_vendor.import_vendor('GOOGLE'); print('GOOGLE backend OK')

## 4. Компиляция под Tensor G5
`keep_going=True` → partial-delegation: что компилятор не возьмёт на TPU, уйдёт в CPU-фолбэк
(именно этого не умел NNAPI). `truncation_type='half'` → fp16 на TPU.

In [ ]:
from ai_edge_litert.aot import aot_compile as aot_lib
from ai_edge_litert.aot.vendors.google_tensor import target as gt_target

assert os.path.exists(MODEL_TFLITE), f'нет модели: {MODEL_TFLITE} (выполни Приложение)'
t = gt_target.Target(gt_target.SocModel.TENSOR_G5)

compiled = aot_lib.aot_compile(
    MODEL_TFLITE,
    target=[t],
    keep_going=True,
    google_tensor_truncation_type='half',
)
print(compiled.compilation_report())

## 5. Сохраняем скомпилированную модель на Drive
Появятся: `unet_g5_<backend>.tflite` (для TPU) и `unet_g5_fallback.tflite` (CPU-эталон для сверки).

In [ ]:
import glob
compiled.export(OUT_DIR, model_name='unet_g5')
produced = glob.glob(os.path.join(OUT_DIR, 'unet_g5*'))
for p in produced:
    print(os.path.getsize(p) // 1_000_000, 'MB ', p)
print('\nГотово. Файлы на Drive в', OUT_DIR, '- скинь их мне для прогона на Pixel 10.')

---
## Приложение: сгенерировать `unet.tflite` (fp32) прямо в Colab
**Выполни это ПЕРВЫМ, в чистом рантайме** (`litert-torch` конфликтует с AOT-nightly).
Скачает SD1.5 UNet с HF, сконвертит в fp32-tflite, положит на Drive.
Потом **Runtime → Restart** и иди с ячейки 1 (Приложение больше не запускай).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!pip install -q torch diffusers transformers accelerate litert-torch

import torch, torch.nn as nn, os
from diffusers import UNet2DConditionModel
import litert_torch as L

WORK = '/content/drive/MyDrive/tensor_tpu'; os.makedirs(WORK, exist_ok=True)
unet = UNet2DConditionModel.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', subfolder='unet').float().eval()

class W(nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, lat, ts, ctx): return s.m(lat, ts, ctx, return_dict=False)[0]

m = W(unet).eval()
args = (torch.randn(1,4,64,64), torch.full((1,), 999.0), torch.randn(1,77,768))
L.convert(m, args).export(os.path.join(WORK, 'unet.tflite'))   # fp32, без quant_config
print('unet.tflite (fp32) на Drive. Теперь Runtime → Restart и иди с ячейки 1.')